In [ ]:
"""
Flower Auction House & Bid Tracking Dashboard
Single-file runnable Python desktop application combining SQLite database and Tkinter GUI.
Theme: sky blue and pink, with little flowers.
"""

import math
import sqlite3
from datetime import date
from tkinter import *
from tkinter import messagebox
from tkinter import ttk

# ==========================================
# DATABASE CLASS
# ==========================================

class Database:
    def __init__(self, db):
        self.con = sqlite3.connect(db)
        self.cur = self.con.cursor()
        # SQLite ignores foreign keys unless this is switched ON for the connection.
        # Without it, ON DELETE CASCADE would silently do nothing.
        self.cur.execute("PRAGMA foreign_keys = ON")

        self.cur.execute("""
        CREATE TABLE IF NOT EXISTS auction_houses(
            house_id INTEGER PRIMARY KEY AUTOINCREMENT,
            house_name TEXT NOT NULL UNIQUE COLLATE NOCASE,
            contact_email TEXT NOT NULL
        );
        """)
        self.cur.execute("""
        CREATE TABLE IF NOT EXISTS auction_lots(
            lot_id INTEGER PRIMARY KEY AUTOINCREMENT,
            house_id INTEGER NOT NULL,
            flower_variety TEXT NOT NULL,
            lot_date TEXT NOT NULL,
            FOREIGN KEY (house_id) REFERENCES auction_houses(house_id) ON DELETE CASCADE
        );
        """)
        self.cur.execute("""
        CREATE TABLE IF NOT EXISTS bid_records(
            bid_id INTEGER PRIMARY KEY AUTOINCREMENT,
            lot_id INTEGER NOT NULL,
            bid_type TEXT NOT NULL,
            bid_value REAL NOT NULL,
            FOREIGN KEY (lot_id) REFERENCES auction_lots(lot_id) ON DELETE CASCADE
        );
        """)
        self.con.commit()
        self.seed()

    # Seed sample data only if all three tables are completely empty
    def seed(self):
        total = 0
        for table in ("auction_houses", "auction_lots", "bid_records"):
            self.cur.execute("SELECT COUNT(*) FROM " + table)
            total += self.cur.fetchone()[0]
        if total > 0:
            return

        self.cur.execute("INSERT INTO auction_houses VALUES (NULL, 'Aalsmeer Flora Exchange', 'auctioneer@aalsmeer.example')")
        self.cur.execute("INSERT INTO auction_houses VALUES (NULL, 'Bengaluru Bloom Mart', 'desk@blrbloom.example')")
        self.cur.execute("INSERT INTO auction_houses VALUES (NULL, 'Pune Petal Auctions', 'bids@punepetal.example')")

        self.cur.execute("INSERT INTO auction_lots VALUES (NULL, 1, 'Red Rose - Grade A', '2026-11-02')")
        self.cur.execute("INSERT INTO auction_lots VALUES (NULL, 1, 'White Chrysanthemum', '2026-11-03')")
        self.cur.execute("INSERT INTO auction_lots VALUES (NULL, 2, 'Yellow Marigold', '2026-11-04')")
        self.cur.execute("INSERT INTO auction_lots VALUES (NULL, 3, 'Pink Carnation - Grade B', '2026-11-05')")

        bids = [
            (1, "Opening Bid", 120), (1, "Final Bid", 185.5), (1, "Quantity Sold", 400),
            (2, "Opening Bid", 80), (2, "Final Bid", 110),
            (3, "Opening Bid", 40), (3, "Quantity Sold", 950),
            (4, "Opening Bid", 65), (4, "Final Bid", 92.25),
        ]
        for b in bids:
            self.cur.execute("INSERT INTO bid_records VALUES (NULL, ?, ?, ?)", b)
        self.con.commit()

    # Insert: reuse house if it exists, else create it; then add 1 lot + 1 bid
    def insert(self, house_name, flower, bid_type, bid_value):
        self.cur.execute("SELECT house_id FROM auction_houses WHERE house_name = ?", (house_name,))
        row = self.cur.fetchone()
        if row:
            house_id = row[0]
        else:
            # email is required by the table but not on the form, so use a placeholder
            placeholder = "auctioneer@" + "".join(c for c in house_name.lower() if c.isalnum()) + ".example"
            self.cur.execute(
                "INSERT INTO auction_houses VALUES (NULL, ?, ?)", (house_name, placeholder)
            )
            house_id = self.cur.lastrowid

        self.cur.execute(
            "INSERT INTO auction_lots VALUES (NULL, ?, ?, ?)",
            (house_id, flower, date.today().isoformat()),
        )
        lot_id = self.cur.lastrowid

        self.cur.execute(
            "INSERT INTO bid_records VALUES (NULL, ?, ?, ?)", (lot_id, bid_type, bid_value)
        )
        self.con.commit()  # one commit: all three inserts succeed together

    # Combined JOIN view
    def fetch_joined(self):
        self.cur.execute("""
            SELECT l.lot_id, h.house_name, l.flower_variety, l.lot_date, b.bid_type, b.bid_value
            FROM auction_lots l
            JOIN auction_houses h ON h.house_id = l.house_id
            JOIN bid_records b ON b.lot_id = l.lot_id
            ORDER BY l.lot_id, b.bid_id
        """)
        return self.cur.fetchall()

    # Raw table views
    def fetch_houses(self):
        self.cur.execute("SELECT * FROM auction_houses")
        return self.cur.fetchall()

    def fetch_lots(self):
        self.cur.execute("SELECT * FROM auction_lots")
        return self.cur.fetchall()

    def fetch_bids(self):
        self.cur.execute("SELECT * FROM bid_records")
        return self.cur.fetchall()

    # Delete functions (children are removed automatically by ON DELETE CASCADE)
    def remove_house(self, house_id):
        self.cur.execute("DELETE FROM auction_houses WHERE house_id=?", (house_id,))
        self.con.commit()

    def remove_lot(self, lot_id):
        self.cur.execute("DELETE FROM auction_lots WHERE lot_id=?", (lot_id,))
        self.con.commit()

    def remove_bid(self, bid_id):
        self.cur.execute("DELETE FROM bid_records WHERE bid_id=?", (bid_id,))
        self.con.commit()


# ==========================================
# COLOURS (sky blue + pink theme)
# ==========================================

SKY = "#bfe6fb"
SKY_LIGHT = "#e6f5fd"
SKY_DARK = "#5bb5e8"
PINK = "#ff8fb8"
PINK_DARK = "#e8639a"
PINK_LIGHT = "#ffe1ee"
TEXT = "#3b4a6b"
FLOWERS = "\u273F  \u2740  \u273E  \u273F  \u2740  \u273E  \u273F  \u2740  \u273E  \u273F  \u2740  \u273E"

# ==========================================
# GUI APPLICATION SETUP
# ==========================================

db = Database("FlowerAuction.db")

root = Tk()
root.title("FLOWER AUCTION HOUSE")
root.geometry("1280x750+50+50")
root.config(bg=SKY)

# Form Variables
house_name = StringVar()
flower = StringVar()
bid_type = StringVar()
bid_value = StringVar()
delete_type = StringVar(value="Auction House")
delete_choice = StringVar()
delete_ids = []  # ids matching the items currently shown in the delete dropdown

# ttk styling (the "clam" theme lets us change colours)
style = ttk.Style()
style.theme_use("clam")
style.configure("mystyle.Treeview", font=("Calibri", 13), rowheight=28,
                background="white", fieldbackground="white", foreground=TEXT)
style.configure("mystyle.Treeview.Heading", font=("Calibri", 14, "bold"),
                background=PINK, foreground="white", relief="flat")
style.map("mystyle.Treeview", background=[("selected", SKY_DARK)],
          foreground=[("selected", "white")])
style.map("mystyle.Treeview.Heading", background=[("active", PINK_DARK)])
style.configure("TNotebook", background=SKY, borderwidth=0)
style.configure("TNotebook.Tab", background=PINK_LIGHT, foreground=TEXT,
                font=("Calibri", 13, "bold"), padding=(16, 6))
style.map("TNotebook.Tab", background=[("selected", PINK)],
          foreground=[("selected", "white")])
style.configure("TCombobox", fieldbackground="white", background=PINK_LIGHT,
                arrowcolor=PINK_DARK, foreground=TEXT)
root.option_add("*TCombobox*Listbox.font", ("Calibri", 14))

# ---------------- Title ----------------
title = Label(
    root,
    text="\u273F  FLOWER AUCTION HOUSE  \u273F",
    font=("Calibri", 24, "bold"),
    bg=SKY_DARK,
    fg="white",
    pady=8,
)
title.pack(side=TOP, fill=X)

Label(root, text=FLOWERS, font=("Calibri", 14), bg=SKY, fg=PINK_DARK).pack(side=TOP, fill=X)

# ---------------- Insert Panel ----------------
entries_frame = LabelFrame(
    root, text="  \u2740  Add a New Bid  \u2740  ", font=("Calibri", 15, "bold"),
    bg=SKY_LIGHT, fg=PINK_DARK, bd=2, relief="groove",
)
entries_frame.pack(side=TOP, fill=X, padx=10, pady=(5, 5))

lblHouse = Label(entries_frame, text="Auction House Name", font=("Calibri", 15), bg=SKY_LIGHT, fg=TEXT)
lblHouse.grid(row=0, column=0, padx=10, pady=8, sticky="w")
txtHouse = Entry(entries_frame, textvariable=house_name, font=("Calibri", 15), width=30,
                 bd=0, highlightthickness=2, highlightbackground=PINK_LIGHT, highlightcolor=PINK)
txtHouse.grid(row=0, column=1, padx=10, pady=8, sticky="w")

lblFlower = Label(entries_frame, text="Flower Variety", font=("Calibri", 15), bg=SKY_LIGHT, fg=TEXT)
lblFlower.grid(row=0, column=2, padx=10, pady=8, sticky="w")
txtFlower = Entry(entries_frame, textvariable=flower, font=("Calibri", 15), width=30,
                  bd=0, highlightthickness=2, highlightbackground=PINK_LIGHT, highlightcolor=PINK)
txtFlower.grid(row=0, column=3, padx=10, pady=8, sticky="w")

lblBidType = Label(entries_frame, text="Bid Type", font=("Calibri", 15), bg=SKY_LIGHT, fg=TEXT)
lblBidType.grid(row=1, column=0, padx=10, pady=8, sticky="w")
comboBidType = ttk.Combobox(entries_frame, font=("Calibri", 15), width=28, textvariable=bid_type)
comboBidType["values"] = ("Opening Bid", "Final Bid", "Quantity Sold")
comboBidType.grid(row=1, column=1, padx=10, pady=8, sticky="w")

lblBidValue = Label(entries_frame, text="Bid Value", font=("Calibri", 15), bg=SKY_LIGHT, fg=TEXT)
lblBidValue.grid(row=1, column=2, padx=10, pady=8, sticky="w")
txtBidValue = Entry(entries_frame, textvariable=bid_value, font=("Calibri", 15), width=30,
                    bd=0, highlightthickness=2, highlightbackground=PINK_LIGHT, highlightcolor=PINK)
txtBidValue.grid(row=1, column=3, padx=10, pady=8, sticky="w")


def make_button(parent, text, command, width=16):
    # Pink button that gets darker when the mouse is over it
    btn = Button(parent, text=text, command=command, width=width,
                 font=("Calibri", 14, "bold"), fg="white", bg=PINK,
                 activebackground=PINK_DARK, activeforeground="white",
                 bd=0, cursor="hand2")
    btn.bind("<Enter>", lambda event: btn.config(bg=PINK_DARK))
    btn.bind("<Leave>", lambda event: btn.config(bg=PINK))
    return btn


# ---------------- Event Functions ----------------

def fill(tree, rows):
    # Insert rows with alternating (zebra) colours
    for i, r in enumerate(rows):
        tag = "odd" if i % 2 else "even"
        tree.insert("", END, values=r, tags=(tag,))


def displayAll():
    # Refresh all four tabs from the database
    for tree in (tv_join, tv_houses, tv_lots, tv_bids):
        tree.delete(*tree.get_children())
    fill(tv_join, db.fetch_joined())
    fill(tv_houses, db.fetch_houses())
    fill(tv_lots, db.fetch_lots())
    fill(tv_bids, db.fetch_bids())


def loadDeleteOptions():
    # Rebuild the dropdown from the database so it never shows an ID that no longer exists
    global delete_ids
    kind = delete_type.get()
    labels = []
    delete_ids = []

    if kind == "Auction House":
        for r in db.fetch_houses():
            labels.append("House " + str(r[0]) + " - " + str(r[1]))
            delete_ids.append(r[0])
    elif kind == "Auction Lot":
        for r in db.fetch_lots():
            labels.append("Lot " + str(r[0]) + " - " + str(r[2]) + " (" + str(r[3]) + ")")
            delete_ids.append(r[0])
    else:
        for r in db.fetch_bids():
            labels.append("Bid " + str(r[0]) + " - Lot " + str(r[1]) + " - " + str(r[2]) + ": " + str(r[3]))
            delete_ids.append(r[0])

    comboDelete["values"] = labels
    if labels:
        comboDelete.current(0)
    else:
        delete_choice.set("")


def clearAll():
    house_name.set("")
    flower.set("")
    bid_type.set("")
    bid_value.set("")
    displayAll()
    loadDeleteOptions()


def add_record():
    if (
        txtHouse.get().strip() == ""
        or txtFlower.get().strip() == ""
        or comboBidType.get().strip() == ""
        or txtBidValue.get().strip() == ""
    ):
        messagebox.showerror("Error in Input", "Please Fill All the Details")
        return

    try:
        value = float(txtBidValue.get().strip())
    except ValueError:
        messagebox.showerror("Error in Input", "Bid Value must be a number")
        return

    if not math.isfinite(value):
        messagebox.showerror("Error in Input", "Bid Value must be a valid number")
        return
    if value < 0:
        messagebox.showerror("Error in Input", "Bid Value cannot be negative")
        return

    db.insert(
        txtHouse.get().strip(),
        txtFlower.get().strip(),
        comboBidType.get().strip(),
        value,
    )
    messagebox.showinfo("Success", "Record Inserted \u273F")
    clearAll()


def delete_selected():
    index = comboDelete.current()
    if index == -1 or index >= len(delete_ids):
        messagebox.showwarning("Selection Required", "Please select an item to delete.")
        return

    kind = delete_type.get()
    item_id = delete_ids[index]

    if kind == "Auction House":
        message = "Delete this house and ALL its lots and bid records?"
    elif kind == "Auction Lot":
        message = "Delete this lot and ALL its bid records?"
    else:
        message = "Delete this bid record only?"
    if not messagebox.askyesno("Confirm Delete", message):
        return

    if kind == "Auction House":
        db.remove_house(item_id)
    elif kind == "Auction Lot":
        db.remove_lot(item_id)
    else:
        db.remove_bid(item_id)

    messagebox.showinfo("Success", "Record Deleted")
    clearAll()


# ---------------- Add Button ----------------
make_button(entries_frame, "Add Record \u2740", add_record).grid(
    row=2, column=0, padx=10, pady=8, sticky="w"
)

# ---------------- Delete Panel ----------------
delete_frame = LabelFrame(
    root, text="  \u273E  Delete  \u273E  ", font=("Calibri", 15, "bold"),
    bg=SKY_LIGHT, fg=PINK_DARK, bd=2, relief="groove",
)
delete_frame.pack(side=TOP, fill=X, padx=10, pady=5)

Label(delete_frame, text="Delete a:", font=("Calibri", 15), bg=SKY_LIGHT, fg=TEXT).grid(
    row=0, column=0, padx=10, pady=8
)
comboType = ttk.Combobox(
    delete_frame,
    font=("Calibri", 15),
    width=15,
    textvariable=delete_type,
    state="readonly",
)
comboType["values"] = ("Auction House", "Auction Lot", "Bid Record")
comboType.grid(row=0, column=1, padx=10, pady=8)
comboType.bind("<<ComboboxSelected>>", lambda event: loadDeleteOptions())

comboDelete = ttk.Combobox(
    delete_frame,
    font=("Calibri", 15),
    width=50,
    textvariable=delete_choice,
    state="readonly",
)
comboDelete.grid(row=0, column=2, padx=10, pady=8)

make_button(delete_frame, "Delete Selected", delete_selected).grid(
    row=0, column=3, padx=10, pady=8
)

# ---------------- Footer ----------------
Label(root, text=FLOWERS, font=("Calibri", 14), bg=SKY, fg=PINK_DARK).pack(side=BOTTOM, fill=X)

# ---------------- Tabbed Display ----------------
notebook = ttk.Notebook(root)
notebook.pack(side=TOP, fill=BOTH, expand=True, padx=10, pady=10)


def make_tab(tab_title, columns):
    # Creates one tab containing a Treeview with scrollbars and returns the Treeview
    frame = Frame(notebook, bg=SKY_LIGHT)
    notebook.add(frame, text=tab_title)

    tree = ttk.Treeview(frame, columns=columns, style="mystyle.Treeview", selectmode="browse")
    for c in columns:
        tree.heading(c, text=c)
        tree.column(c, width=150, anchor="center")
    tree["show"] = "headings"
    tree.tag_configure("even", background="white")
    tree.tag_configure("odd", background="#fff0f6")

    vsb = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
    hsb = ttk.Scrollbar(frame, orient="horizontal", command=tree.xview)
    tree.configure(yscrollcommand=vsb.set, xscrollcommand=hsb.set)

    tree.pack(side=LEFT, fill=BOTH, expand=True)
    vsb.pack(side=RIGHT, fill=Y)
    hsb.pack(side=BOTTOM, fill=X)
    return tree


tv_join = make_tab("\u273F Combined View", ("Lot ID", "House Name", "Flower Variety", "Lot Date", "Bid Type", "Bid Value"))
tv_houses = make_tab("\u2740 Auction Houses", ("House ID", "House Name", "Contact Email"))
tv_lots = make_tab("\u273E Auction Lots", ("Lot ID", "House ID", "Flower Variety", "Lot Date"))
tv_bids = make_tab("\u273F Bid Records", ("Bid ID", "Lot ID", "Bid Type", "Bid Value"))

# Initial load
displayAll()
loadDeleteOptions()

if __name__ == "__main__":
    root.mainloop()